# Medical prognosis based on FL + genetic algorithms

In [1]:
import gensim.downloader as api

corpus = api.load('glove-wiki-gigaword-100')

In [6]:
import pandas as pd

data = pd.read_csv('../resources/medical/disease_diagnosis.csv')
data['symptomps_text'] = ""

with open("../resources/medical/symptoms_text_2.txt", "r", encoding="utf-8") as f:
    for line in f:
        index, text = line.strip().split("||")
        index = int(index)
        data.at[index, "symptomps_text"] = text

data.to_csv('../resources/medical/disease_diagnosis_with_symptoms.csv', index=False)
data.head(10)

,Patient_ID,Age,Gender,Symptom_1,Symptom_2,Symptom_3,Heart_Rate_bpm,Body_Temperature_C,Blood_Pressure_mmHg,Oxygen_Saturation_%,Diagnosis,Severity,Treatment_Plan,symptomps_text
0,1,74,Male,Fatigue,Sore throat,Fever,69,39.4,132/91,94,Flu,Moderate,Medication and rest,I've been feeling really off lately. It starte...
1,2,66,Female,Sore throat,Fatigue,Cough,95,39.0,174/98,98,Healthy,Mild,Rest and fluids,I've been feeling really unwell lately. It sta...
2,3,32,Male,Body ache,Sore throat,Fatigue,77,36.8,136/60,96,Healthy,Mild,Rest and fluids,I've been feeling really unwell lately. It sta...
3,4,21,Female,Shortness of breath,Headache,Cough,72,38.9,147/82,99,Healthy,Mild,Rest and fluids,I've been feeling quite unwell lately. It star...
4,5,53,Male,Runny nose,Sore throat,Fatigue,100,36.6,109/106,92,Healthy,Mild,Rest and fluids,I've been feeling really unwell lately. It sta...
5,6,22,Male,Sore throat,Fever,Cough,90,39.5,107/92,93,Flu,Moderate,Medication and rest,"For the past few days, I’ve been feeling prett..."
6,7,21,Male,Sore throat,Fatigue,Cough,71,37.5,126/82,93,Bronchitis,Severe,Hospitalization and medication,I've been feeling really unwell lately. It sta...
7,8,71,Male,Headache,Shortness of breath,Runny nose,64,38.6,153/99,99,Healthy,Mild,Rest and fluids,"For the past few days, I've been feeling quite..."
8,9,56,Female,Shortness of breath,Fever,Headache,103,36.2,152/71,96,Cold,Mild,Rest and fluids,I've been feeling really unwell lately. It sta...
9,10,53,Male,Cough,Fever,Headache,62,39.5,111/104,98,Flu,Moderate,Medication and rest,"For the past few days, I've been feeling reall..."


In [15]:
import random

data['Blood_Pressure_mmHg_lower'] = data['Blood_Pressure_mmHg'].apply(lambda x: int(x.split('/')[1]) if isinstance(x, str) and '/' in x else None)
data['Blood_Pressure_mmHg_upper'] = data['Blood_Pressure_mmHg'].apply(lambda x: int(x.split('/')[0]) if isinstance(x, str) and '/' in x else None)
data["Gender_encoded"] = data["Gender"].map({"Male": 0, "Female": 1})

feature_list = ["Age", "Heart_Rate_bpm", "Body_Temperature_C", "Blood_Pressure_mmHg_upper", "Blood_Pressure_mmHg_lower", "Oxygen_Saturation_%"]
feature_list_scaled = []

for item in feature_list:
    data[item + "_scaled"] = (data[item] - data[item].min()) / (data[item].max() - data[item].min())
    feature_list_scaled.append(item + "_scaled")

feature_list_scaled.append("Gender_encoded")

# data[feature_list_scaled].head()
# print("Feature list (scaled):", feature_list_scaled)

level = ["low", "medium", "high"]
severity = ["mild", "moderate", "severe"]

population_size = 100
rules_size = 10

def get_random_rule():
    return random.choice(feature_list_scaled) + " " + random.choice(level) + " and " + random.choice(feature_list_scaled) + " " + random.choice(level) + " then " + random.choice(severity)

initial_rule = "".join([f + " medium and " for f in feature_list_scaled])
initial_rule = initial_rule[:-4] + "then moderate"


flrb = [initial_rule] + [get_random_rule() for _ in range(rules_size - 1)]
print(flrb)

['Age_scaled medium and Heart_Rate_bpm_scaled medium and Body_Temperature_C_scaled medium and Blood_Pressure_mmHg_upper_scaled medium and Blood_Pressure_mmHg_lower_scaled medium and Oxygen_Saturation_%_scaled medium and Gender_encoded medium then moderate', 'Body_Temperature_C_scaled medium and Body_Temperature_C_scaled low then mild', 'Blood_Pressure_mmHg_upper_scaled medium and Body_Temperature_C_scaled high then severe', 'Heart_Rate_bpm_scaled high and Body_Temperature_C_scaled high then mild', 'Body_Temperature_C_scaled high and Blood_Pressure_mmHg_upper_scaled medium then severe', 'Blood_Pressure_mmHg_lower_scaled low and Blood_Pressure_mmHg_lower_scaled medium then mild', 'Blood_Pressure_mmHg_lower_scaled low and Body_Temperature_C_scaled medium then moderate', 'Blood_Pressure_mmHg_lower_scaled low and Gender_encoded low then moderate', 'Oxygen_Saturation_%_scaled high and Heart_Rate_bpm_scaled medium then moderate', 'Oxygen_Saturation_%_scaled high and Blood_Pressure_mmHg_lower_